In [1]:
!pip install supabase python-dotenv pandas numpy scipy PySastrawi


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =========================================================
# CELL 1 - IMPORT LIBRARY + KONEKSI SUPABASE
# =========================================================
import os
import re
import math
import json
import pickle
import sys
import scipy.sparse as sp
import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

BACKEND_DIR = next(
    path for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "backend")
    if (path / "src" / "preprocessing").is_dir()
)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from src.preprocessing.stopwords import get_stopwords

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

SOURCE_TABLE = "cleaned_papers_results"
TFIDF_TABLE = "tfidf_weights"

print("✅ Supabase client siap")

✅ Supabase client siap


In [3]:
# =========================================================
# CELL 2 - LOAD DATA DARI CLEANED PAPERS
# =========================================================
print("📥 Mengambil data dari Supabase...")

all_data = []
batch_size = 1000
offset = 0

while True:
    response = (
        supabase.table(SOURCE_TABLE)
        .select("id, title, abstract,category")
        .range(offset, offset + batch_size - 1)
        .execute()
    )

    batch = response.data or []

    if not batch:
        break

    all_data.extend(batch)
    print(f"  → Batch {offset // batch_size + 1}: {len(batch)} data")

    if len(batch) < batch_size:
        break

    offset += batch_size

df = pd.DataFrame(all_data)

if df.empty:
    raise ValueError("❌ Data kosong. Cek tabel cleaned_papers_results.")

df["id"] = df["id"].astype("int64")
df["title"] = df["title"].fillna("").astype(str)
df["abstract"] = df["abstract"].fillna("").astype(str)
df["category"] = df["category"].fillna("unknown").astype(str)

print(f"✅ Total artikel: {len(df)}")
print("Distribusi kategori:")
print(df["category"].value_counts(dropna=False))
df.head(3)

📥 Mengambil data dari Supabase...
  → Batch 1: 200 data
✅ Total artikel: 200
Distribusi kategori:
category
machine learning      50
cyber security        50
mobile application    50
web application       50
Name: count, dtype: int64


,id,title,abstract,category
0,1,Penerapan machine learning dalam prediksi ting...,… menjabarkan implementasi machine learning un...,machine learning
1,2,Tinjauan Pustaka Sistematis: Penerapan Metode ...,… Machine Learning dapat mempelajari pola data...,machine learning
2,3,"Penerapan Machine Learning, Deep Learning, Dan...","… the application of machine learning, deep le...",machine learning


In [4]:
# =========================================================
# CELL 3 - STOPWORDS + STEMMER
# =========================================================
stop_words = get_stopwords()

stemmer = StemmerFactory().create_stemmer()

print("✅ Stopwords dan stemmer siap")

✅ Stopwords dan stemmer siap


In [5]:
# =========================================================
# CELL 4 - FUNGSI PREPROCESSING TITLE + ABSTRACT
# =========================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_to_tokens(text):
    cleaned = clean_text(text)
    tokens = cleaned.split()
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    tokens = [stemmer.stem(token) for token in tokens]
    return tokens

print("✅ Fungsi preprocessing siap")

✅ Fungsi preprocessing siap


In [6]:
# =========================================================
# CELL 5 - MEMBUAT DOKUMEN VSM: TITLE + ABSTRACT
# =========================================================
df["title_tokens"] = df["title"].apply(preprocess_to_tokens)
df["abstract_tokens"] = df["abstract"].apply(preprocess_to_tokens)

# Sesuai proposal: judul dan abstrak digabung menjadi satu dokumen
df["document_tokens"] = df["title_tokens"] + df["abstract_tokens"]
df["document_text"] = df["document_tokens"].apply(lambda tokens: " ".join(tokens))

print("✅ Dokumen VSM dari title + abstract selesai dibuat")
print("Dokumen kosong:", int(df["document_text"].eq("").sum()))

df[["id", "category", "title", "document_text"]].head(5)

✅ Dokumen VSM dari title + abstract selesai dibuat
Dokumen kosong: 0


,id,category,title,document_text
0,1,machine learning,Penerapan machine learning dalam prediksi ting...,terap machine learning prediksi tingkat kasus ...
1,2,machine learning,Tinjauan Pustaka Sistematis: Penerapan Metode ...,tinjau pustaka sistematis terap metode machine...
2,3,machine learning,"Penerapan Machine Learning, Deep Learning, Dan...",terap machine learning deep learning data mini...
3,4,machine learning,Penerapan machine learning untuk prediksi benc...,terap machine learning prediksi bencana banjir...
4,5,machine learning,Penerapan Machine Learning dan Deep Learning p...,terap machine learning deep learning tingkat d...


In [7]:
# =========================================================
# CELL 6 - AUDIT DOKUMEN TF-IDF
# =========================================================
df["len_title_tokens"] = df["title_tokens"].apply(len)
df["len_abstract_tokens"] = df["abstract_tokens"].apply(len)
df["len_document_tokens"] = df["document_tokens"].apply(len)

print("Rata-rata token title:", round(df["len_title_tokens"].mean(), 2))
print("Rata-rata token abstract:", round(df["len_abstract_tokens"].mean(), 2))
print("Rata-rata token gabungan:", round(df["len_document_tokens"].mean(), 2))
print("Dokumen kosong:", int((df["len_document_tokens"] == 0).sum()))

df[["category", "title", "title_tokens", "abstract_tokens", "document_tokens"]].sample(
    min(5, len(df)),
    random_state=42
)

Rata-rata token title: 8.9
Rata-rata token abstract: 18.14
Rata-rata token gabungan: 27.04
Dokumen kosong: 0


,category,title,title_tokens,abstract_tokens,document_tokens
95,cyber security,Artificial intelligence in cyber security,"[artificial, intelligence, cyber, security]","[security, preparation, instance, cyber, secur...","[artificial, intelligence, cyber, security, se..."
15,machine learning,Penerapan Machine Learning menggunakan algorit...,"[terap, machine, learning, guna, algoritma, c4...","[didik, sangat, pegang, peran, penting, tingka...","[terap, machine, learning, guna, algoritma, c4..."
30,machine learning,Financial machine learning,"[financial, machine, learning]","[we, survey, nascent, literature, machine, lea...","[financial, machine, learning, we, survey, nas..."
158,web application,Perancangan Aplikasi Pemesanan Makanan Berbasi...,"[ancang, aplikasi, mesan, makan, bas, web]","[bas, web, aplikasi, buat, rancang, menu, maka...","[ancang, aplikasi, mesan, makan, bas, web, bas..."
128,mobile application,Why people choose Apps: An evaluation of the e...,"[why, people, choose, apps, evaluation, ecolog...","[mobile, applications, seventy, nine, responde...","[why, people, choose, apps, evaluation, ecolog..."


In [8]:
# =========================================================
# CELL 7 - HITUNG DF DAN IDF
# Rumus proposal:
# IDF = log10(N / df_t)
# =========================================================
N = len(df)

document_frequency = Counter()

for tokens in df["document_tokens"]:
    unique_terms = set(tokens)
    document_frequency.update(unique_terms)

idf_scores = {
    term: math.log10(N / df_value)
    for term, df_value in document_frequency.items()
    if df_value > 0
}

terms = sorted(idf_scores.keys())

print("✅ IDF selesai dihitung")
print("Jumlah dokumen:", N)
print("Jumlah term:", len(terms))

pd.DataFrame([
    {
        "term": term,
        "df": document_frequency[term],
        "idf": idf_scores[term]
    }
    for term in terms[:10]
])

✅ IDF selesai dihitung
Jumlah dokumen: 200
Jumlah term: 1732


,term,df,idf
0,10,1,2.301030
1,12,1,2.301030
2,16,2,2.000000
3,17,1,2.301030
4,19,7,1.455932
5,1959,1,2.301030
6,20,1,2.301030
7,2019,1,2.301030
8,2030,1,2.301030
9,276,1,2.301030


In [9]:
# =========================================================
# CELL 8 - HITUNG TF-IDF
# Rumus:
# TF = jumlah kemunculan term / total kata dokumen
# TF-IDF = TF × IDF
# =========================================================
tfidf_records = []
tfidf_detail_records = []

ts = datetime.now(timezone.utc).isoformat()

for _, row in df.iterrows():
    doc_id = int(row["id"])
    title = row["title"]
    category = row["category"]
    tokens = row["document_tokens"]
    total_terms = len(tokens)

    if total_terms == 0:
        continue

    term_counts = Counter(tokens)

    for term, count in term_counts.items():
        tf = count / total_terms
        idf = idf_scores.get(term, 0)
        tfidf_score = tf * idf

        tfidf_records.append({
            "doc_id": doc_id,
            "category": category,
            "title": title,
            "term": term,
            "tfidf_score": float(tfidf_score),
            "updated_at": ts
        })

        tfidf_detail_records.append({
            "doc_id": doc_id,
            "category": category,   
            "title": title,
            "term": term,
            "term_count": count,
            "total_terms": total_terms,
            "tf": tf,
            "df": document_frequency[term],
            "idf": idf,
            "tfidf_score": tfidf_score
        })

tfidf_df = pd.DataFrame(tfidf_detail_records)

print("✅ TF-IDF selesai dihitung")
print("Total bobot TF-IDF:", len(tfidf_records))

tfidf_df.head(10)

✅ TF-IDF selesai dihitung
Total bobot TF-IDF: 3910


,doc_id,category,title,term,term_count,total_terms,tf,df,idf,tfidf_score
0,1,machine learning,Penerapan machine learning dalam prediksi ting...,terap,1,23,0.043478,30,0.823909,0.035822
1,1,machine learning,Penerapan machine learning dalam prediksi ting...,machine,4,23,0.173913,54,0.568636,0.098893
2,1,machine learning,Penerapan machine learning dalam prediksi ting...,learning,4,23,0.173913,63,0.501689,0.087250
3,1,machine learning,Penerapan machine learning dalam prediksi ting...,prediksi,2,23,0.086957,9,1.346787,0.117112
4,1,machine learning,Penerapan machine learning dalam prediksi ting...,tingkat,1,23,0.043478,17,1.070581,0.046547
5,1,machine learning,Penerapan machine learning dalam prediksi ting...,kasus,2,23,0.086957,8,1.397940,0.121560
6,1,machine learning,Penerapan machine learning dalam prediksi ting...,sakit,1,23,0.043478,4,1.698970,0.073868
7,1,machine learning,Penerapan machine learning dalam prediksi ting...,indonesia,1,23,0.043478,19,1.022276,0.044447
8,1,machine learning,Penerapan machine learning dalam prediksi ting...,jabar,1,23,0.043478,1,2.301030,0.100045
9,1,machine learning,Penerapan machine learning dalam prediksi ting...,implementasi,2,23,0.086957,7,1.455932,0.126603


In [10]:
# =========================================================
# CELL 9 - BENTUK MATRIX VSM
# Baris = artikel
# Kolom = term
# Nilai = bobot TF-IDF
# =========================================================
doc_ids = df["id"].astype(int).tolist()
term_to_index = {term: idx for idx, term in enumerate(terms)}
doc_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_ids)}

rows = []
cols = []
values = []

for record in tfidf_detail_records:
    rows.append(doc_to_index[record["doc_id"]])
    cols.append(term_to_index[record["term"]])
    values.append(record["tfidf_score"])

tfidf_matrix = sp.csr_matrix(
    (values, (rows, cols)),
    shape=(len(doc_ids), len(terms))
)

print("✅ Matrix VSM selesai dibuat")
print("Ukuran matrix:", tfidf_matrix.shape)
print("Total nilai non-zero:", tfidf_matrix.nnz)

✅ Matrix VSM selesai dibuat
Ukuran matrix: (200, 1732)
Total nilai non-zero: 3910


In [11]:
# =========================================================
# CELL 10 - SIMPAN FILE LOKAL TF-IDF + VSM
# =========================================================
save_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "tfidf"))
os.makedirs(save_dir, exist_ok=True)

tfidf_detail_path = os.path.join(save_dir, "tfidf_weights_detail.csv")
tfidf_weights_path = os.path.join(save_dir, "tfidf_weights.csv")
documents_path = os.path.join(save_dir, "tfidf_documents.csv")
matrix_path = os.path.join(save_dir, "tfidf_matrix.npz")
terms_path = os.path.join(save_dir, "tfidf_terms.json")
doc_ids_path = os.path.join(save_dir, "tfidf_doc_ids.json")
idf_path = os.path.join(save_dir, "idf_scores.json")

tfidf_df.to_csv(tfidf_detail_path, index=False)
pd.DataFrame(tfidf_records).to_csv(tfidf_weights_path, index=False)
df[
    [
        "id",
        "category",
        "title",
        "abstract",
        "title_tokens",
        "abstract_tokens",
        "document_tokens",
        "document_text"
    ]
].to_csv(documents_path, index=False)

sp.save_npz(matrix_path, tfidf_matrix)

with open(terms_path, "w", encoding="utf-8") as file:
    json.dump(terms, file, ensure_ascii=False, indent=2)

with open(doc_ids_path, "w", encoding="utf-8") as file:
    json.dump(doc_ids, file, ensure_ascii=False, indent=2)

with open(idf_path, "w", encoding="utf-8") as file:
    json.dump(idf_scores, file, ensure_ascii=False, indent=2)

print("✅ File lokal TF-IDF tersimpan di:", save_dir)

✅ File lokal TF-IDF tersimpan di: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf


In [12]:
# =========================================================
# CELL 11 - SIMULASI MANUAL TITLE DAN ABSTRACT TERPISAH
# Untuk penjelasan laporan/proposal
# Tidak wajib masuk ke tabel tfidf_weights
# =========================================================
manual_records = []

for _, row in df.iterrows():
    doc_id = int(row["id"])
    title = row["title"]
    category = row["category"]

    parts = {
        "title": row["title_tokens"],
        "abstract": row["abstract_tokens"]
    }

    for document_part, tokens in parts.items():
        total_terms = len(tokens)

        if total_terms == 0:
            continue

        term_counts = Counter(tokens)

        for term, count in term_counts.items():
            tf = count / total_terms
            idf = idf_scores.get(term, 0)
            tfidf_score = tf * idf

            manual_records.append({
                "doc_id": doc_id,
                "category": category,
                "title": title,
                "document_part": document_part,
                "term": term,
                "term_count": count,
                "total_terms": total_terms,
                "tf": tf,
                "df": document_frequency.get(term, 0),
                "idf": idf,
                "tfidf_score": tfidf_score
            })

manual_df = pd.DataFrame(manual_records)

manual_path = os.path.join(save_dir, "tfidf_manual_title_abstract.csv")
manual_df.to_csv(manual_path, index=False)

print("✅ Simulasi manual title/abstract tersimpan:", manual_path)
manual_df.head(10)

✅ Simulasi manual title/abstract tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\tfidf_manual_title_abstract.csv


,doc_id,category,title,document_part,term,term_count,total_terms,tf,df,idf,tfidf_score
0,1,machine learning,Penerapan machine learning dalam prediksi ting...,title,terap,1,8,0.125000,30,0.823909,0.102989
1,1,machine learning,Penerapan machine learning dalam prediksi ting...,title,machine,1,8,0.125000,54,0.568636,0.071080
2,1,machine learning,Penerapan machine learning dalam prediksi ting...,title,learning,1,8,0.125000,63,0.501689,0.062711
3,1,machine learning,Penerapan machine learning dalam prediksi ting...,title,prediksi,1,8,0.125000,9,1.346787,0.168348
4,1,machine learning,Penerapan machine learning dalam prediksi ting...,title,tingkat,1,8,0.125000,17,1.070581,0.133823
5,1,machine learning,Penerapan machine learning dalam prediksi ting...,title,kasus,1,8,0.125000,8,1.397940,0.174743
6,1,machine learning,Penerapan machine learning dalam prediksi ting...,title,sakit,1,8,0.125000,4,1.698970,0.212371
7,1,machine learning,Penerapan machine learning dalam prediksi ting...,title,indonesia,1,8,0.125000,19,1.022276,0.127785
8,1,machine learning,Penerapan machine learning dalam prediksi ting...,abstract,jabar,1,15,0.066667,1,2.301030,0.153402
9,1,machine learning,Penerapan machine learning dalam prediksi ting...,abstract,implementasi,2,15,0.133333,7,1.455932,0.194124


In [13]:
# =========================================================
# CELL 12 - CONTOH VSM MANUAL UNTUK 1 ARTIKEL
# =========================================================
example_doc_id = int(df.iloc[0]["id"])

example_terms = (
    tfidf_df[tfidf_df["doc_id"] == example_doc_id]
    .sort_values("tfidf_score", ascending=False)
    .head(5)["term"]
    .tolist()
)

example_vsm = manual_df[
    (manual_df["doc_id"] == example_doc_id) &
    (manual_df["term"].isin(example_terms))
].pivot_table(
    index=["doc_id", "document_part"],
    columns="term",
    values="tfidf_score",
    fill_value=0
).reset_index()

print("Contoh artikel ID:", example_doc_id)
print("5 term utama:", example_terms)

example_vsm

Contoh artikel ID: 1
5 term utama: ['implementasi', 'kasus', 'prediksi', 'jabar', 'arima']


term,doc_id,document_part,arima,implementasi,jabar,kasus,prediksi
0,1,abstract,0.153402,0.194124,0.153402,0.093196,0.089786
1,1,title,0.000000,0.000000,0.000000,0.174743,0.168348


In [14]:
# =========================================================
# CELL 13 - HELPER INSERT SUPABASE
# =========================================================
def insert_batches(table_name, records, batch_size=500):
    total = len(records)

    if total == 0:
        print(f"⚠️ Tidak ada data untuk dimasukkan ke {table_name}")
        return

    for start in range(0, total, batch_size):
        batch = records[start:start + batch_size]

        supabase.table(table_name).insert(batch).execute()

        print(f"  → Batch {start // batch_size + 1}: {len(batch)} data")

    print(f"✅ Insert {total} data ke {table_name}")

In [15]:
# =========================================================
# CELL 14 - INSERT TF-IDF KE SUPABASE
# =========================================================
insert_batches(
    table_name=TFIDF_TABLE,
    records=tfidf_records,
    batch_size=500
)

check_tfidf = (
    supabase.table(TFIDF_TABLE)
    .select("doc_id", count="exact")
    .limit(1)
    .execute()
)

print("✅ Jumlah data di tfidf_weights:", check_tfidf.count)

  → Batch 1: 500 data
  → Batch 2: 500 data
  → Batch 3: 500 data
  → Batch 4: 500 data
  → Batch 5: 500 data
  → Batch 6: 500 data
  → Batch 7: 500 data
  → Batch 8: 410 data
✅ Insert 3910 data ke tfidf_weights
✅ Jumlah data di tfidf_weights: 3910


In [16]:
# =========================================================
# CELL 15 - OUTPUT VSM READABLE
# Hati-hati: bisa besar, tapi berguna untuk inspeksi
# =========================================================
vsm_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=doc_ids,
    columns=terms
)

vsm_df.index.name = "doc_id"

vsm_readable_path = os.path.join(save_dir, "vsm_matrix_readable.csv")
vsm_df.to_csv(vsm_readable_path)

print("✅ VSM readable tersimpan:", vsm_readable_path)
print("Ukuran VSM:", vsm_df.shape)

vsm_df.head()

✅ VSM readable tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\vsm_matrix_readable.csv
Ukuran VSM: (200, 1732)


,10,12,16,17,19,1959,20,2019,2030,276,...,writing,wujud,xss,xyz,years,you,young,your,zap,zone
doc_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
# =========================================================
# CELL 16 - SAMPLE VSM PER KATEGORI
# Ini pengganti sample lama yang cuma head(10)
# =========================================================
sample_ids = []

for category, group in df.groupby("category"):
    sample_ids.extend(group["id"].head(3).astype(int).tolist())

sample_indices = [
    doc_ids.index(doc_id)
    for doc_id in sample_ids
    if doc_id in doc_ids
]

sample_matrix = tfidf_matrix[sample_indices]
sample_doc_ids = [doc_ids[index] for index in sample_indices]

top_terms_sample = (
    pd.DataFrame({
        "term": terms,
        "total_weight": sample_matrix.sum(axis=0).A1
    })
    .query("total_weight > 0")
    .sort_values("total_weight", ascending=False)
    .head(15)["term"]
    .tolist()
)

vsm_sample_df = pd.DataFrame(
    sample_matrix.toarray(),
    index=sample_doc_ids,
    columns=terms
)

vsm_sample_df = vsm_sample_df[top_terms_sample]
vsm_sample_df.index.name = "doc_id"

category_lookup = df.set_index("id")["category"].to_dict()
vsm_sample_df.insert(
    0,
    "category",
    [category_lookup.get(doc_id, "") for doc_id in sample_doc_ids]
)

vsm_sample_path = os.path.join(save_dir, "vsm_matrix_sample.csv")
vsm_sample_df.to_csv(vsm_sample_path)

print("✅ Sample VSM per kategori tersimpan:", vsm_sample_path)
print("Term yang ditampilkan:", top_terms_sample)

vsm_sample_df

✅ Sample VSM per kategori tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\vsm_matrix_sample.csv
Term yang ditampilkan: ['aman', 'aplikasi', 'siber', 'tren', 'learning', 'pemrograman', 'ajar', 'machine', 'kembang', 'tingkat', 'hasil', 'tinjau', 'privasi', 'web', 'programming']


,category,aman,aplikasi,siber,tren,learning,pemrograman,ajar,machine,kembang,tingkat,hasil,tinjau,privasi,web,programming
doc_id,,,,,,,,,,,,,,,,
51,cyber security,0.172017,0.000000,0.129013,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.050980,0.000000,0.000000,0.00000,0.000000,0.000000
52,cyber security,0.129013,0.000000,0.161266,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
53,cyber security,0.157059,0.000000,0.117794,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.069655,0.20009,0.000000,0.000000
1,machine learning,0.000000,0.024723,0.000000,0.000000,0.087250,0.000000,0.000000,0.098893,0.000000,0.046547,0.000000,0.000000,0.00000,0.000000,0.000000
2,machine learning,0.000000,0.000000,0.000000,0.000000,0.077183,0.000000,0.061618,0.087482,0.000000,0.000000,0.051800,0.061618,0.00000,0.000000,0.000000
3,machine learning,0.000000,0.000000,0.000000,0.000000,0.121622,0.000000,0.000000,0.068926,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
101,mobile application,0.000000,0.103388,0.000000,0.313777,0.000000,0.000000,0.000000,0.000000,0.194651,0.000000,0.000000,0.072821,0.00000,0.000000,0.000000
102,mobile application,0.000000,0.087482,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.041176,0.000000,0.000000,0.00000,0.000000,0.000000
103,mobile application,0.000000,0.081234,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.038235,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000


In [18]:
# =========================================================
# CELL 17 - TOP TERM PER KATEGORI
# Untuk membuktikan 4 kategori punya bobot masing-masing
# =========================================================
top_category_rows = []

for category, group in df.groupby("category"):
    category_doc_ids = group["id"].astype(int).tolist()
    category_indices = [
        doc_ids.index(doc_id)
        for doc_id in category_doc_ids
        if doc_id in doc_ids
    ]

    if not category_indices:
        continue

    category_matrix = tfidf_matrix[category_indices]

    category_terms_df = pd.DataFrame({
        "category": category,
        "term": terms,
        "total_weight": category_matrix.sum(axis=0).A1,
        "document_count": (category_matrix > 0).sum(axis=0).A1
    })

    category_terms_df = (
        category_terms_df[category_terms_df["total_weight"] > 0]
        .sort_values("total_weight", ascending=False)
        .head(10)
    )

    top_category_rows.append(category_terms_df)

top_terms_per_category_df = pd.concat(top_category_rows, ignore_index=True)

top_terms_per_category_path = os.path.join(save_dir, "top_terms_per_category.csv")
top_terms_per_category_df.to_csv(top_terms_per_category_path, index=False)

print("✅ Top term per kategori tersimpan:", top_terms_per_category_path)

top_terms_per_category_df

✅ Top term per kategori tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\top_terms_per_category.csv


,category,term,total_weight,document_count
0,cyber security,aman,3.380564,25
1,cyber security,siber,3.268513,25
2,cyber security,cyber,3.227229,29
3,cyber security,security,2.426225,25
4,cyber security,cybersecurity,0.807456,10
5,cyber security,digital,0.632986,9
6,cyber security,indonesia,0.592897,11
7,cyber security,era,0.513280,6
8,cyber security,sistem,0.458433,7
9,cyber security,strategi,0.442281,4


In [19]:
# =========================================================
# CELL 18 - VALIDASI TERM KATEGORI UTAMA
# =========================================================
category_terms = [
    "machine", "learning",
    "web", "develop", "development",
    "cyber", "security", "secure",
    "mobile", "application", "app", "apps"
]

available_terms = [term for term in category_terms if term in terms]
missing_terms = [term for term in category_terms if term not in terms]

print("Term tersedia:", available_terms)
print("Term tidak ada:", missing_terms)

term_summary_df = pd.DataFrame({
    "term": available_terms,
    "document_count": [
        int((tfidf_matrix[:, terms.index(term)] > 0).sum())
        for term in available_terms
    ],
    "total_weight": [
        float(tfidf_matrix[:, terms.index(term)].sum())
        for term in available_terms
    ]
}).sort_values("total_weight", ascending=False)

term_summary_path = os.path.join(save_dir, "category_term_summary.csv")
term_summary_df.to_csv(term_summary_path, index=False)

term_summary_df

Term tersedia: ['machine', 'learning', 'web', 'develop', 'development', 'cyber', 'security', 'mobile', 'application', 'app', 'apps']
Term tidak ada: ['secure']


,term,document_count,total_weight
1,learning,63,3.796315
0,machine,54,3.479750
7,mobile,54,3.398391
5,cyber,29,3.227229
2,web,48,2.955102
6,security,31,2.679448
8,application,38,1.794710
10,apps,19,1.435952
9,app,15,1.106665
4,development,9,0.478214
